### Python packages used in this code

In [ ]:
import importlib
import glob
import natsort
import sys
utils_path = './'
data_path = './data'
if utils_path not in sys.path:
    sys.path.append(utils_path)

from MyPackages import *
from MyFunctions import *
from MyTorch import *
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms = True
MyDevice = set_device('')

%matplotlib inline
MySEED = 373



def fun_Reload(pkg):
    module = importlib.import_module(pkg)
    module = importlib.reload(module)
    attrs = {attr: getattr(module, attr) for attr in dir(module)}
    globals().update(attrs)
    
fun_Reload('MyPackages')
fun_Reload('MyFunctions')
fun_Reload('MyTorch')


print('--Platform--')
print('OS :', platform.platform())
print('--Version--')
print('python : ', sys.version)
print('numpy :', np.__version__)
print('pandas :', pd.__version__)

## Environments
--Platform--

OS : Linux-6.5.11-linuxkit-x86_64-with-glibc2.31

--Version--

python :  3.9.7 | packaged by conda-forge | (default, Sep 29 2021, 19:23:11) 

[GCC 9.4.0]

numpy : 1.26.4

pandas : 2.0.3

In [ ]:
## Read virtual moleculs SMILES
df_smiles = pd.read_csv('./data/Virtual/virtual_mol.csv', index_col=0)

## Descroptor parameter
mp = 10 # parallel number
n = 10 # cyclic oligomerization for polymer
nk = 20 # number of Gaussian in discretization (Hyperparameter)
sigma = 1/nk/2 # band width of Gaussian kernel (Hyperparameter)
mu = None # list of discretizing points (Hyperparameter)
ff_desc = FF_descriptor(GAFF2_mod(), polar =True)

df_vir = ff_desc.ffkm_mp(df_smiles['SMILES'].to_list(), mp =mp, nk =nk, s=sigma, mu =mu, cyclic=n)

In [ ]:
source_prop = {"Tm": "TC_bond",
             "Tiso": "compressibility",
             "Class":"Rg"}

dim = 190
inter_dim_1 = 512
inter_dim_2 = 256
inter_dim_3 = 128
inter_dim_4 = 64
inter_dim_5 = 32

class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear_1 = nn.Linear(dim, inter_dim_1)
        self.linear_2 = nn.Linear(inter_dim_1, inter_dim_2)
        self.linear_3 = nn.Linear(inter_dim_2, inter_dim_3)
        self.linear_4 = nn.Linear(inter_dim_3, inter_dim_4)
        
        self.linear_5_NSC = nn.Linear(inter_dim_4, inter_dim_5)
        self.linear_5_LC = nn.Linear(inter_dim_4, inter_dim_5)
        self.linear_out_NSC = nn.Linear(inter_dim_5, 3)
        self.linear_out_LC = nn.Linear(inter_dim_5, 1)
        
    def forward(self, x):
        x = F.relu(self.linear_1(x))
        x = F.relu(self.linear_2(x))
        x = F.relu(self.linear_3(x))
        x = F.relu(self.linear_4(x))
        x_NSC = F.relu(self.linear_5_NSC(x))
        out_NSC = self.linear_out_NSC(x_NSC)
        x_LC = F.relu(self.linear_5_LC(x))
        out_LC = self.linear_out_LC(x_LC)
        
        return out_NSC, out_LC

Class_model = Model()

Tm_model = MyModel(
    input_dim = 190, 
    layers_dict = {
        'fc1': 512, 
        'fc2': 256,
        'fc3': 128,
        'fc4': 64,
        'fc5': 32,
        'fc6': 1
    }
)

Tiso_model = MyModel(
    input_dim = 190, 
    layers_dict = {
        'fc1': 512, 
        'fc2': 256,
        'fc3': 128,
        'fc4': 64,
        'fc5': 32,
        'fc6': 1
    }
)

with open(f'./Result/{source_prop["Tm"]}/Target_Tm_Model/all.pickle', 'rb') as f:
    Tm_dict = pickle.load(f)
with open(f'./Result/{source_prop["Tiso"]}/Target_Tiso_Model/all.pickle', 'rb') as f:
    Tiso_dict = pickle.load(f)
with open(f'./Result/{source_prop["Class"]}/Target_Class_Model/all.pickle', 'rb') as f:
    Class_dict = pickle.load(f)

Tm_Source_number = natsort.natsorted(glob.glob(f'./Result/{source_prop["Tm"]}/Target_Tm_Model/*.pth'))[-1].split('Source')[1].split('.')[0]
Tiso_Source_number = natsort.natsorted(glob.glob(f'./Result/{source_prop["Tiso"]}/Target_Tiso_Model/*.pth'))[-1].split('Source')[1].split('.')[0]
Class_Source_number = natsort.natsorted(glob.glob(f'./Result/{source_prop["Class"]}/Target_Class_Model/*.pth'))[-1].split('Source')[1].split('.')[0]

df_list = []
for n_seed in tqdm(range(3)):
    Tm_dict_seed = Tm_dict[f'Seed{n_seed}NumSource{Tm_Source_number}']
    Tiso_dict_seed = Tiso_dict[f'Seed{n_seed}NumSource{Tiso_Source_number}']
    Class_dict_seed = Class_dict[f'Seed{n_seed}NumSource{Class_Source_number}']

    scaled_X_class = Class_dict_seed['scaler_x'].transform(df_vir)
    scaled_X_Tm = Tm_dict_seed['scaler_x'].transform(df_vir)
    scaled_X_Tiso = Tiso_dict_seed['scaler_x'].transform(df_vir)

    Class_model.load_state_dict(Class_dict_seed['model'])
    Tm_model.load_state_dict(Tm_dict_seed['model'])
    Tiso_model.load_state_dict(Tiso_dict_seed['model'])

    pred_y_class = pd.DataFrame(sigmoid(Class_model(torch.Tensor(scaled_X_class.values).float())[0].detach().numpy()),
                                columns=['N', 'S', 'C'], index=df_vir.index)
    scaled_pred_y_Tm = Tm_model(torch.Tensor(scaled_X_Tm.values).float()).detach().numpy()
    scaled_pred_y_Tiso = Tiso_model(torch.Tensor(scaled_X_Tiso.values).float()).detach().numpy()
    pred_y_Tm = Tm_dict_seed['scaler_y'].inverse(pd.DataFrame(scaled_pred_y_Tm,columns=['Tm']))
    pred_y_Tm.index = df_vir.index
    pred_y_Tiso = Tiso_dict_seed['scaler_y'].inverse(pd.DataFrame(scaled_pred_y_Tiso,columns=['Tiso']))
    pred_y_Tiso.index = df_vir.index

    alpha = 0.5  # nominal coverage 1 - alpha = 50 %
    w_qhat_N = weight_split_CP(pd.concat([Class_dict_seed['train_x_NSC'],Class_dict_seed['cal_x_NSC'],scaled_X_class]),
                            Class_dict_seed['cal_idx_NSC'],
                            Class_dict_seed['train_idx_NSC'],
                            df_vir.index.to_list(), alpha,
                                Class_dict_seed['cal_y_NSC'].values[:,0],
                                Class_dict_seed['cal_pred_y_NSC'].values[:,0])
    w_qhat_S = weight_split_CP(pd.concat([Class_dict_seed['train_x_NSC'],Class_dict_seed['cal_x_NSC'],scaled_X_class]),
                            Class_dict_seed['cal_idx_NSC'],
                            Class_dict_seed['train_idx_NSC'],
                            df_vir.index.to_list(), alpha,
                                Class_dict_seed['cal_y_NSC'].values[:,1],
                                Class_dict_seed['cal_pred_y_NSC'].values[:,1])
    w_qhat_C = weight_split_CP(pd.concat([Class_dict_seed['train_x_NSC'],Class_dict_seed['cal_x_NSC'],scaled_X_class]),
                            Class_dict_seed['cal_idx_NSC'],
                            Class_dict_seed['train_idx_NSC'],
                            df_vir.index.to_list(), alpha,
                                Class_dict_seed['cal_y_NSC'].values[:,0],
                                Class_dict_seed['cal_pred_y_NSC'].values[:,2])
    w_qhat_Tm = weight_split_CP(pd.concat([Tm_dict_seed['train_x'],Tm_dict_seed['cal_x'],scaled_X_Tm]),
                            Tm_dict_seed['cal_idx'],
                            Tm_dict_seed['train_idx'],
                            df_vir.index.to_list(), alpha,
                                Tm_dict_seed['cal_y'].values,
                                Tm_dict_seed['cal_pred_y'].values)
    w_qhat_Tiso = weight_split_CP(pd.concat([Tiso_dict_seed['train_x'],Tiso_dict_seed['cal_x'],scaled_X_Tiso]),
                            Tiso_dict_seed['cal_idx'],
                            Tiso_dict_seed['train_idx'],
                            df_vir.index.to_list(), alpha,
                                Tiso_dict_seed['cal_y'].values,
                                Tiso_dict_seed['cal_pred_y'].values)

    df_qhat = pd.DataFrame(np.stack([w_qhat_N, w_qhat_S, w_qhat_C, w_qhat_Tm, w_qhat_Tiso], 1),
                        columns=['N_qhat', 'S_qhat', 'C_qhat', 'Tm_qhat', 'Tiso_qhat'], index=df_vir.index)

    output = pd.concat([pred_y_class, pred_y_Tm, pred_y_Tiso, df_qhat], axis=1)
    prop_list = ['N','S','C','Tm','Tiso']
    for prop in prop_list:
        output[f'{prop}_lower'] = output[f'{prop}'] - output[f'{prop}_qhat']
        output[f'{prop}_upper'] = output[f'{prop}'] + output[f'{prop}_qhat']
    df_list.append(output)
df_predict = pd.concat(df_list, axis=0)
df_predict_mean = df_predict.groupby(level=0).mean()
df_predict_mean.to_csv(f'./Result/Virtual_Prediction.csv')

